In [1]:
# Make the notebook wider 
from IPython.display import display, HTML
display(HTML("<style>.container { width:80% !important; }</style>"))

%load_ext autoreload
%autoreload 2

<br>
<div style="text-align: left; font-family: Arial, sans-serif; color: #003366;">
    <font size=9><b>Step 1: Compare Different Models with Different Sets of Input Features</b></font>
</div>
<br>

The objective of this step is to identify the most effective combination of:

<ul>
  <li><b>Input features</b> — it is not guaranteed that using all available predictors improves performance.</li>
  <li><b>Model type</b> — comparing both <b>linear</b> and <b>non-linear</b> regressors.</li>
  <li><b>Hyperparameter values</b> — values that must be set before training a model and that will influence its performance.</li>
</ul>

<br>

To achieve this, we perform an extensive <span style="color:#003366"><b>cross-validation</b></span> procedure. This ensures that model evaluation is <b>robust</b>, <b>comparable</b>, and <b>generalizable</b>, preventing overfitting to any specific data subset.

All numerical variables, as well as the target <span style="color:#003366"><b>Die_area</b></span>, are analyzed in <b>log scale</b>.<br><br>

<div style="background-color: #ffffff; padding: 12px; border-left: 5px solid #003366; margin-top: 10px;">
💡 <b>Note:</b> The cross-validation procedure is performed exclusively on the data from the <b>train set</b>.
</div>
<br><br>


In [3]:
import os
import pandas as pd
import numpy as np
from itertools import combinations

from Utils.loadData import load_data
from Utils.crossValidation import evaluate_models_with_grid_search


#######################################################################################################
# HELPER FUNCTION — Convert parameter dictionary to filename
#######################################################################################################
def param_dict_to_filename(params):
    """
    Convert a parameter dictionary into a descriptive filename.
    Example:
        {"features": ["M", "P_area"]} → "features-[M,P_area].csv"
    """
    parts = []
    for key, value in params.items():
        if isinstance(value, list):
            val_str = "[" + ",".join(map(str, value)) + "]"
        else:
            val_str = str(value)
        parts.append(f"{key}-{val_str}")
    return "_".join(parts) + ".csv"


#######################################################################################################
# MAIN FUNCTION — Run all grid searches and save results
#######################################################################################################
def run_all_grid_searches(df_train, df_test, all_features, resFolder="CVRuns"):
    """
    Run grid searches for all feature combinations and save cross-validation summaries to disk.
    """
    # Generate all possible feature combinations
    feature_combinations = [
        list(c) for k in range(1, len(all_features) + 1)
        for c in combinations(all_features, k)
    ]

    # Iterate over each feature set
    for combi in feature_combinations:
        params = {"features": combi}
        print(f"--- Running grid search for features: {combi}")
        summary_df = evaluate_models_with_grid_search(df_train, **params)
        filename = os.path.join(resFolder, param_dict_to_filename(params))
        summary_df.to_csv(filename, index=False)

    print(f"\nAll grid search results saved in: {resFolder}")


#######################################################################################################
# DATA PREPARATION
#######################################################################################################
df_train, df_test = load_data("Data/dataset.xlsx")
all_features      = ["M", "P_area", "Pin_count", "P_type"]
resFolder         = "CVRuns"  # Output folder
os.makedirs(resFolder, exist_ok=True)


# ----------------------------------------------------------------------------------------------------
# RUN CROSS-VALIDATION
# ----------------------------------------------------------------------------------------------------
# To execute the full benchmark, uncomment the line below. It will train all models across all feature subsets and save their summaries.

#run_all_grid_searches(df_train, df_test, all_features, resFolder=resFolder)


--- Running grid search for features: ['M']
Linear Regression
KNN
Decision Tree
SVR
MLP
--- Running grid search for features: ['P_area']
Linear Regression
KNN
Decision Tree
SVR
MLP
--- Running grid search for features: ['Pin_count']
Linear Regression
KNN
Decision Tree
SVR
MLP
--- Running grid search for features: ['P_type']
Linear Regression
KNN
Decision Tree
SVR
MLP
Group-Wise Linear
--- Running grid search for features: ['M', 'P_area']
Linear Regression
KNN
Decision Tree
SVR
MLP
--- Running grid search for features: ['M', 'Pin_count']
Linear Regression
KNN
Decision Tree
SVR
MLP
--- Running grid search for features: ['M', 'P_type']
Linear Regression
KNN
Decision Tree
SVR
MLP
Group-Wise Linear
--- Running grid search for features: ['P_area', 'Pin_count']
Linear Regression
KNN
Decision Tree
SVR
MLP
--- Running grid search for features: ['P_area', 'P_type']
Linear Regression
KNN
Decision Tree
SVR
MLP
Group-Wise Linear
--- Running grid search for features: ['Pin_count', 'P_type']
Linear R

In [4]:
from Utils.printResults import display_final_table
display_final_table(df_train, df_test, all_features, folder="CVRuns")

,#Train,#Test,M,P_area,Pin_count,P_type,RMSE train (Linear Regression),RMSE val (Linear Regression),RMSE test (Linear Regression),RMSE train (Group-Wise Linear),RMSE val (Group-Wise Linear),RMSE test (Group-Wise Linear),RMSE train (KNN),RMSE val (KNN),RMSE test (KNN),RMSE train (Decision Tree),RMSE val (Decision Tree),RMSE test (Decision Tree),RMSE train (SVR),RMSE val (SVR),RMSE test (SVR),RMSE train (MLP),RMSE val (MLP),RMSE test (MLP),Mean train,Mean val,Mean test
0,297,148,✓,,,,0.5 ± 0.014,0.5 ± 0.17,0.53,nan,nan,nan,0.12 ± 0.0067,0.52 ± 0.14,0.55,0.44 ± 0.011,0.49 ± 0.14,0.66,0.47 ± 0.015,0.49 ± 0.19,0.48,0.46 ± 0.013,0.48 ± 0.17,0.51,0.4,0.5,0.55
1,491,148,,✓,,,0.4 ± 0.021,0.37 ± 0.24,0.42,nan,nan,nan,0.17 ± 0.0055,0.42 ± 0.17,0.47,0.31 ± 0.016,0.39 ± 0.22,0.59,0.39 ± 0.02,0.37 ± 0.22,0.39,0.4 ± 0.021,0.36 ± 0.26,0.4,0.33,0.38,0.45
2,459,148,,,✓,,0.41 ± 0.011,0.4 ± 0.14,0.33,nan,nan,nan,0.35 ± 0.024,0.41 ± 0.14,0.57,0.32 ± 0.0088,0.39 ± 0.11,0.35,0.39 ± 0.01,0.39 ± 0.13,0.33,0.4 ± 0.013,0.4 ± 0.14,0.33,0.37,0.4,0.38
3,297,148,✓,,✓,,0.37 ± 0.012,0.36 ± 0.16,0.32,nan,nan,nan,0.045 ± 0.0043,0.36 ± 0.16,0.31,0.25 ± 0.0099,0.37 ± 0.13,0.46,0.35 ± 0.013,0.35 ± 0.15,0.32,0.35 ± 0.014,0.34 ± 0.16,0.33,0.27,0.36,0.35
4,297,148,✓,,,✓,0.4 ± 0.0092,0.4 ± 0.13,0.39,0.39 ± 0.0086,0.42 ± 0.14,0.41,0.055 ± 0.0053,0.43 ± 0.14,0.39,0.3 ± 0.012,0.43 ± 0.14,0.47,0.38 ± 0.011,0.48 ± 0.19,0.36,0.36 ± 0.01,0.43 ± 0.13,0.38,0.31,0.43,0.4
5,297,148,✓,✓,,,0.39 ± 0.019,0.37 ± 0.22,0.4,nan,nan,nan,0.034 ± 0.0038,0.39 ± 0.19,0.37,0.25 ± 0.012,0.42 ± 0.19,0.4,0.36 ± 0.018,0.37 ± 0.21,0.37,0.35 ± 0.019,0.39 ± 0.18,0.42,0.28,0.39,0.39
6,459,148,,✓,✓,,0.36 ± 0.017,0.35 ± 0.19,0.32,nan,nan,nan,0.1 ± 0.0045,0.35 ± 0.13,0.32,0.2 ± 0.0088,0.38 ± 0.13,0.45,0.32 ± 0.013,0.35 ± 0.16,0.29,0.34 ± 0.013,0.32 ± 0.16,0.31,0.26,0.35,0.34
7,491,148,,✓,,✓,0.33 ± 0.013,0.34 ± 0.15,0.27,0.31 ± 0.012,0.33 ± 0.14,0.3,0.15 ± 0.0037,0.37 ± 0.11,0.36,0.19 ± 0.0064,0.38 ± 0.11,0.42,0.31 ± 0.012,0.33 ± 0.13,0.3,0.31 ± 0.012,0.33 ± 0.14,0.28,0.27,0.35,0.32
8,459,148,,,✓,✓,0.41 ± 0.012,0.42 ± 0.15,0.33,0.39 ± 0.012,0.41 ± 0.15,0.34,0.28 ± 0.01,0.39 ± 0.13,0.43,0.29 ± 0.009,0.41 ± 0.12,0.36,0.38 ± 0.012,0.39 ± 0.15,0.34,0.39 ± 0.011,0.41 ± 0.15,0.33,0.36,0.41,0.35
9,297,148,✓,✓,,✓,0.32 ± 0.014,0.33 ± 0.16,0.27,0.31 ± 0.014,0.35 ± 0.18,0.29,0.033 ± 0.0041,0.39 ± 0.15,0.36,0.15 ± 0.0094,0.43 ± 0.16,0.45,0.29 ± 0.013,0.35 ± 0.14,0.3,0.29 ± 0.011,0.35 ± 0.16,0.3,0.23,0.37,0.33


<div style="border: 2px solid steelblue; border-radius: 8px; padding: 15px; background-color: #f0f8ff; font-family: Arial, sans-serif;">

<b>Understanding RMSE — The Root Mean Squared Error</b><br><br>

The <b>Root Mean Squared Error (RMSE)</b> is the primary metric used to evaluate model performance in this notebook.  
It quantifies the average magnitude of prediction errors — that is, how far the predicted values are from the observed ones.

<br><br>

The RMSE is defined as:

<div style="text-align:center; font-size: 16px; margin-top:10px; margin-bottom:10px;">
  $$ \mathrm{RMSE} = \sqrt{ \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2 } $$
</div>

$$
\text{where } \hat{y}_i \text{ and } y_i \text{ represent the predicted and observed values, respectively.}
$$

<ul>
  <li><b>Lower RMSE</b> values indicate better model performance.</li>
  <li><b>RMSE = 0</b> means perfect prediction (no deviation from observed data).</li>
  <li>Because RMSE is expressed in the same units as the target variable, it directly reflects the prediction accuracy.</li>
</ul>

<br>

<b>RMSE in Log-Log Space</b><br>

This means RMSE measures the <b>average deviation between predicted and true log(Die area)</b> values.  
Because the model operates in logarithmic space, the RMSE represents a <b>multiplicative error</b> on the original scale:

<ul>
  <li>An RMSE of <b>0.1</b> corresponds to predictions differing by roughly a factor of 10⁰·¹ ≈ <b>1.26</b>.</li>
  <li>An RMSE of <b>0.3</b> corresponds to a factor of 10⁰·³ ≈ <b>2</b>, meaning predicted die areas differ by about a factor of two on average.</li>
</ul>

<div style="background-color: #ffffff; padding: 12px; border-left: 5px solid #cc5500; margin-top: 10px;">
  ⚠️ <b>Important:</b> RMSE in log space emphasizes <i>relative accuracy</i> rather than absolute error.  
  A model with low log-RMSE provides consistent proportional predictions even if large dies have larger absolute differences.
</div>

</div>
<br><br>
